# 02构建订单级分析宽表

本阶段目标是将不同粒度的原始业务表整理为订单级分析宽表order_base。根据01数据概览的结果，orders表可作为订单主表；order_items、payments和reviews表存在一单多行情况，因此本阶段先将这些表聚合到order_id粒度，再与orders合并，构建订单级分析宽表order_base。

## 1 读取数据

In [28]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data_raw")
OUTPUT_DIR = Path("../data_clean")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("当前工作目录：", Path.cwd())
print("数据目录：", DATA_DIR.resolve())
print("保存目录：", OUTPUT_DIR.resolve())

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")

当前工作目录： d:\olist-analysis\notebooks
数据目录： D:\olist-analysis\data_raw
保存目录： D:\olist-analysis\data_clean


## 2 处理订单主表orders

In [19]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

order_base = orders.copy()

order_base["delivery_days"] = ((
    order_base["order_delivered_customer_date"]
    - order_base["order_purchase_timestamp"]
).dt.total_seconds() / 86400).round(2)

order_base["delay_days"] = ((
    order_base["order_delivered_customer_date"]
    - order_base["order_estimated_delivery_date"]
).dt.total_seconds() / 86400).round(2)

order_base["is_delivered"] = order_base["order_status"] == "delivered"
order_base["is_late"] = order_base["delay_days"] > 0

display(order_base.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delay_days,is_delivered,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.44,-7.11,True,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.78,-5.36,True,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.39,-17.25,True,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.21,-12.98,True,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.87,-9.24,True,False


**观察与总结**

delivery_days表示从下单到实际送达的天数，delay_days表示实际送达时间与预计送达时间的差值。delay_days大于0表示延迟送达，小于等于0表示按时或提前送达。

## 3 聚合reviews评价表

In [20]:
reviews_agg = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "count"),
    )
)

display(reviews_agg.head())

,order_id,review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


## 4 聚合payments支付表

In [21]:
main_payment_type = (
    payments
    .sort_values(
        ["order_id", "payment_value", "payment_sequential"],
        ascending=[True, False, True],
    )
    .drop_duplicates("order_id")
    [["order_id", "payment_type"]]
    .rename(columns={"payment_type": "main_payment_type"})
)  # 对于存在多笔支付的订单，将payment_value最高的支付记录对应的payment_type定义为main_payment_type。若同一订单中存在多笔相同最高金额支付，则按payment_sequential升序取更早的一笔支付记录，保证主支付方式口径稳定。

payments_agg = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        payment_total=("payment_value", "sum"),
        payment_count=("payment_sequential", "count"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_count=("payment_type", "nunique"),
    )
    .merge(main_payment_type, on="order_id", how="left")
)

display(payments_agg.head())

,order_id,payment_total,payment_count,payment_installments_max,payment_type_count,main_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2,1,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3,1,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5,1,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2,1,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3,1,credit_card


## 5 聚合订单商品明细表

In [22]:
items_agg = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        price_total=("price", "sum"),
        freight_total=("freight_value", "sum"),
    )
)

display(items_agg.head())

,order_id,item_count,product_count,seller_count,price_total,freight_total
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14


## 6 合并各个聚合表，生成order_base订单分析宽表

### 6.1 合并前覆盖率检查

判断主表orders里的order_id与各聚合表的order_id的匹配情况

In [23]:
merge_coverage = pd.DataFrame(
    {
        "source_table": [
            "customers",
            "reviews_agg",
            "payments_agg",
            "items_agg",
        ],
        "orders_total": [
            orders["order_id"].nunique()
        ] * 4,
        "matched_orders": [
            orders["customer_id"].isin(customers["customer_id"]).sum(),
            orders["order_id"].isin(reviews_agg["order_id"]).sum(),
            orders["order_id"].isin(payments_agg["order_id"]).sum(),
            orders["order_id"].isin(items_agg["order_id"]).sum(),
        ],
    }
)

merge_coverage["unmatched_orders"] = (
    merge_coverage["orders_total"] - merge_coverage["matched_orders"]
)

merge_coverage["matched_rate"] = (
    merge_coverage["matched_orders"] / merge_coverage["orders_total"]
).round(4)

display(merge_coverage)

,source_table,orders_total,matched_orders,unmatched_orders,matched_rate
0,customers,99441,99441,0,1.0000
1,reviews_agg,99441,98673,768,0.9923
2,payments_agg,99441,99440,1,1.0000
3,items_agg,99441,98666,775,0.9922


**观察与总结**

orders表与customers表可以通过customer_id完全匹配；orders表中的order_id也基本可以在payments_agg中匹配。reviews_agg和items_agg与orders相比存在少量未匹配订单，说明部分订单缺少评价或商品明细，需要结合订单状态进一步判断。

### 6.2 主表合并

In [24]:
order_base = (
    order_base
    .merge(customers, on="customer_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(items_agg, on="order_id", how="left")
)

print("order_base行数:", order_base.shape[0])
print("orders行数:", orders.shape[0])
print("order_base列数:", order_base.shape[1])
print("order_id是否唯一:", order_base["order_id"].is_unique)

display(order_base.head())

order_base行数: 99441
orders行数: 99441
order_base列数: 28
order_id是否唯一: True


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delay_days,...,payment_total,payment_count,payment_installments_max,payment_type_count,main_payment_type,item_count,product_count,seller_count,price_total,freight_total
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.44,-7.11,...,38.71,3.0,1.0,2.0,voucher,1.0,1.0,1.0,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.78,-5.36,...,141.46,1.0,1.0,1.0,boleto,1.0,1.0,1.0,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.39,-17.25,...,179.12,1.0,3.0,1.0,credit_card,1.0,1.0,1.0,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.21,-12.98,...,72.20,1.0,1.0,1.0,credit_card,1.0,1.0,1.0,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.87,-9.24,...,28.62,1.0,1.0,1.0,credit_card,1.0,1.0,1.0,19.90,8.72


### 6.3 关键字段缺失检查

In [25]:
key_cols = [
    "customer_unique_id",
    "review_score",
    "payment_total",
    "price_total",
    "freight_total",
]

order_base_missing = (
    order_base[key_cols]
    .isna()
    .sum()
    .to_frame("missing_count")
)

order_base_missing["missing_rate"] = (
    order_base_missing["missing_count"] / len(order_base)
).round(4)

display(order_base_missing)

,missing_count,missing_rate
customer_unique_id,0,0.0000
review_score,768,0.0077
payment_total,1,0.0000
price_total,775,0.0078
freight_total,775,0.0078


**观察与总结**

review_score来自reviews的聚合表，price_total与freight_total来着order_items的聚合表，因此这些字段的缺失行数与6.1中聚合表的缺失情况一致

### 6.4 缺失原因分析

主要判断缺失值与订单状态的关系

In [26]:
missing_by_status = (
    order_base
    .assign(
        missing_customer=order_base["customer_unique_id"].isna(),
        missing_review=order_base["review_score"].isna(),
        missing_payment=order_base["payment_total"].isna(),
        missing_items=order_base["price_total"].isna(),
    )
    .groupby("order_status", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        missing_customer=("missing_customer", "sum"),
        missing_review=("missing_review", "sum"),
        missing_payment=("missing_payment", "sum"),
        missing_items=("missing_items", "sum"),
    )
)

for col in [
    "missing_customer",
    "missing_review",
    "missing_payment",
    "missing_items",
]:
    missing_by_status[f"{col}_rate"] = (
        missing_by_status[col] / missing_by_status["orders"]
    ).round(4)

display(missing_by_status)

,order_status,orders,missing_customer,missing_review,missing_payment,missing_items,missing_customer_rate,missing_review_rate,missing_payment_rate,missing_items_rate
0,approved,2,0,0,0,0,0.0,0.0000,0.0,0.0000
1,canceled,625,0,20,0,164,0.0,0.0320,0.0,0.2624
2,created,5,0,2,0,5,0.0,0.4000,0.0,1.0000
3,delivered,96478,0,646,1,0,0.0,0.0067,0.0,0.0000
4,invoiced,314,0,5,0,2,0.0,0.0159,0.0,0.0064
5,processing,301,0,6,0,0,0.0,0.0199,0.0,0.0000
6,shipped,1107,0,75,0,1,0.0,0.0678,0.0,0.0009
7,unavailable,609,0,14,0,603,0.0,0.0230,0.0,0.9901


**观察与总结**

缺失字段按订单状态统计后可以看出，items_agg相关字段缺失主要来自unavailable、canceled、created等未完成订单，delivered订单不存在商品明细缺失；review_score在delivered订单中存在少量缺失，表示部分已送达订单没有评价记录；payment_total在delivered订单中存在1条缺失，后续进行销售金额分析时需要排除该订单或单独标记。

### 6.5 导出订单级分析宽表

In [27]:
order_base.to_csv(OUTPUT_DIR / "order_base.csv", index=False)

## 总结

本阶段以orders表为主表，构建了一张订单级分析宽表order_base。由于reviews、payments和order_items表均存在一单多行情况，直接JOIN会导致订单行数膨胀，因此本阶段先将这些表按order_id聚合到订单粒度，再与orders表合并。

在order_base中，每一行代表一个订单，order_id保持唯一。该表整合了订单状态、客户信息、履约时间、评价评分、支付金额和商品明细汇总指标，可作为后续销售表现、履约时效、客户满意度和复购风险分析的基础数据。

本阶段同时构造了delivery_days、delay_days、is_delivered和is_late等履约字段。后续进行物流时效和评分分析时，将主要基于is_delivered为True的订单展开。